# Proposed validation — review before it runs

**What this measures:** `dice_gap` (segformer3d_dice − segresnet_dice) measures whether the real `SegFormer3D` module, trained from scratch under an identical fixed-budget protocol as `SegResNet`, reaches segmentation-accuracy parity with an established MONAI net — now measured on the real, cached Task01_BrainTumour 8/4 subset (fetched once via `monai.apps.DecathlonDataset`) instead of synthetic volumes, per the reviewer's explicit instruction to use the real data and otherwise keep the protocol untouched.

**Target metric:** `dice_gap`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

The executable copy lives at `eval/eval_segformer3d_brats_parity.py`, which is what `.remyx/validation.yaml` points at; keep the two in step, or point `suite:` here if you would rather maintain the notebook.

In [1]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

In [2]:
# Parameters
variant = "feature"
ref = "207c02b1a63f754a0dd51a855835a972e7df7834"
seed = 0


## Execution context

The cells below are the script at `eval/eval_segformer3d_brats_parity.py`, unchanged. This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [3]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "eval/eval_segformer3d_brats_parity.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

[remyx] cwd /workspace/target_repo | script /workspace/target_repo/eval/eval_segformer3d_brats_parity.py


In [4]:
#!/usr/bin/env python
# Copyright (c) MONAI Consortium
# Licensed under the Apache License, Version 2.0 (the "License");
"""
eval/eval_segformer3d_brats_parity.py

Fixed-budget, from-scratch Dice-parity check between SegFormer3D (this PR) and
SegResNet (an existing MONAI net), trained on a fixed real 8/4 subset of the
real Task01_BrainTumour dataset (monai.apps.DecathlonDataset), per
user_guidance. Prints one JSON line with the target metric `dice_gap`, a
guardrail on the shared nets/__init__.py import surface, and cost references.

Runs unchanged on baseline and PR head: SegFormer3D is imported defensively.
On baseline (no SegFormer3D) the SegFormer3D-side metrics degrade to zero and
dice_gap reports a large negative value instead of crashing.

Timing/device metrics follow GPU-tier measurement practice: a few warm-up
iterations are excluded from the timing sample, the reported wall-clock time
is a median-per-iteration extrapolation (not a single noisy elapsed reading),
and peak device memory is read via torch.cuda.max_memory_allocated rather
than sampled.
"""

'\neval/eval_segformer3d_brats_parity.py\n\nFixed-budget, from-scratch Dice-parity check between SegFormer3D (this PR) and\nSegResNet (an existing MONAI net), trained on a fixed real 8/4 subset of the\nreal Task01_BrainTumour dataset (monai.apps.DecathlonDataset), per\nuser_guidance. Prints one JSON line with the target metric `dice_gap`, a\nguardrail on the shared nets/__init__.py import surface, and cost references.\n\nRuns unchanged on baseline and PR head: SegFormer3D is imported defensively.\nOn baseline (no SegFormer3D) the SegFormer3D-side metrics degrade to zero and\ndice_gap reports a large negative value instead of crashing.\n\nTiming/device metrics follow GPU-tier measurement practice: a few warm-up\niterations are excluded from the timing sample, the reported wall-clock time\nis a median-per-iteration extrapolation (not a single noisy elapsed reading),\nand peak device memory is read via torch.cuda.max_memory_allocated rather\nthan sampled.\n'

In [5]:
from __future__ import annotations

import argparse
import itertools
import json
import os
import random
import statistics
import sys
import time

In [6]:
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

SMOKE = os.environ.get("REMYX_SMOKE") == "1"
SEED = 0  # single fixed seed, per user_guidance
CROP = (32, 32, 32) if SMOKE else (64, 64, 64)
N_ITERS = 2 if SMOKE else 200
BATCH = 1 if SMOKE else 2
N_TRAIN = 2 if SMOKE else 8
N_VAL = 1 if SMOKE else 4

In [7]:
parser = argparse.ArgumentParser()
parser.add_argument("--variant", default="")
parser.add_argument("--ref", default="")
parser.add_argument("--seed", default="0")
_ = parser.parse_args()

# ---------------------------------------------------------------------------
# Guardrail: does the shared monai/networks/nets/__init__.py still expose the
# established nets it always has? (independent of SegFormer3D availability)
# ---------------------------------------------------------------------------
EXISTING_NET_NAMES = [
    "UNet", "SegResNet", "BasicUNet", "DynUNet", "AttentionUnet",
    "VNet", "UNETR", "SwinUNETR", "DenseNet121", "VarAutoEncoder",
]
existing_success = 0
try:
    import monai.networks.nets as _nets_mod
    for _name in EXISTING_NET_NAMES:
        try:
            getattr(_nets_mod, _name)
            existing_success += 1
        except Exception:
            pass
except Exception:
    pass
existing_nets_import_success_rate = existing_success / len(EXISTING_NET_NAMES)

/root/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Defensive import of the changed net.
try:
    from monai.networks.nets import SegFormer3D
    HAS_SEGFORMER3D = True
except Exception:
    SegFormer3D = None
    HAS_SEGFORMER3D = False

In [9]:
metrics = {
    "dice_gap": -1.0,
    "existing_nets_import_success_rate": existing_nets_import_success_rate,
    "segformer3d_dice": 0.0,
    "segresnet_dice": 0.0,
    "segformer3d_params": 0,
    "segresnet_params": 0,
    "segformer3d_train_time_s": 0.0,
    "segresnet_train_time_s": 0.0,
    "segformer3d_max_mem_mb": 0.0,
    "segresnet_max_mem_mb": 0.0,
}

In [10]:
def train_model(model, train_loader, device):
    """Train for the fixed N_ITERS step budget (unchanged), while measuring
    wall-clock time and peak device memory the way a GPU-tier eval should:
    exclude a short warm-up from the timing sample, report the median
    per-iteration time (extrapolated to the full budget) rather than a single
    elapsed reading, and read peak memory via max_memory_allocated rather
    than sampling it.
    """
    import torch
    from monai.losses import DiceLoss

    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = DiceLoss(sigmoid=True, squared_pred=True)
    model.train()
    it = itertools.cycle(train_loader)

    if device == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    warmup = min(5, N_ITERS - 1) if N_ITERS > 1 else 0
    iter_times = []

    for step in range(N_ITERS):
        batch = next(it)
        images = batch["image"].to(device)
        labels = batch["label"].to(device)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        opt.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        opt.step()
        if device == "cuda":
            torch.cuda.synchronize()
        dt = time.time() - t0
        if step >= warmup:
            iter_times.append(dt)

    if not iter_times:
        iter_times = [0.0]
    median_iter_time = statistics.median(iter_times)
    total_time_estimate = median_iter_time * N_ITERS

    peak_mem_mb = 0.0
    if device == "cuda":
        peak_mem_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)

    return total_time_estimate, peak_mem_mb

In [11]:
def eval_model(model, val_loader, device):
    import torch
    from monai.metrics import DiceMetric

    model.eval()
    dice_metric = DiceMetric(include_background=True, reduction="mean")
    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)
            preds = (torch.sigmoid(model(images)) > 0.5).float()
            dice_metric(y_pred=preds, y=labels)
    dice = float(dice_metric.aggregate().item())
    dice_metric.reset()
    return dice

In [12]:
try:
    import torch
    from monai.apps import DecathlonDataset
    from monai.data import DataLoader, Dataset
    from monai.networks.nets import SegResNet
    from monai.transforms import (
        Compose, ConvertToMultiChannelBasedOnBratsClassesd, CenterSpatialCropd,
        EnsureChannelFirstd, EnsureTyped, LoadImaged, NormalizeIntensityd,
        Orientationd, SpatialPadd,
    )
    from monai.utils import set_determinism

    device = "cuda" if torch.cuda.is_available() else "cpu"
    set_determinism(seed=SEED)

    data_dir = os.path.join(os.getcwd(), "monai_data")
    os.makedirs(data_dir, exist_ok=True)

    # One-time fetch (download=True is a no-op if the extracted data already
    # exists under data_dir, so repeat invocations reuse the cache).
    raw_ds = DecathlonDataset(
        root_dir=data_dir, task="Task01_BrainTumour", section="training",
        transform=Compose([]), download=True, val_frac=0.0,
        cache_rate=0.0, cache_num=0, num_workers=0, seed=SEED,
    )
    all_files = sorted(raw_ds.data, key=lambda d: str(d["image"]))

    rng = random.Random(SEED)
    order = list(range(len(all_files)))
    rng.shuffle(order)
    train_files = [all_files[i] for i in order[:N_TRAIN]]
    val_files = [all_files[i] for i in order[N_TRAIN:N_TRAIN + N_VAL]]

    transform = Compose([
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        CenterSpatialCropd(keys=["image", "label"], roi_size=CROP),
        SpatialPadd(keys=["image", "label"], spatial_size=CROP),
        NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
        EnsureTyped(keys=["image", "label"], dtype=torch.float32),
    ])

    train_ds = Dataset(data=train_files, transform=transform)
    val_ds = Dataset(data=val_files, transform=transform)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=False, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

    # SegResNet: unaffected by the PR, trained/scored on both arms as a
    # cost reference and as the fixed comparison point for dice_gap.
    set_determinism(seed=SEED)
    segresnet = SegResNet(spatial_dims=3, in_channels=4, out_channels=3, init_filters=16)
    metrics["segresnet_params"] = sum(p.numel() for p in segresnet.parameters())
    metrics["segresnet_train_time_s"], metrics["segresnet_max_mem_mb"] = train_model(
        segresnet, train_loader, device
    )
    metrics["segresnet_dice"] = eval_model(segresnet, val_loader, device)

    if HAS_SEGFORMER3D:
        set_determinism(seed=SEED)
        segformer3d = SegFormer3D(in_channels=4, out_channels=3)
        metrics["segformer3d_params"] = sum(p.numel() for p in segformer3d.parameters())
        metrics["segformer3d_train_time_s"], metrics["segformer3d_max_mem_mb"] = train_model(
            segformer3d, train_loader, device
        )
        metrics["segformer3d_dice"] = eval_model(segformer3d, val_loader, device)
        metrics["dice_gap"] = metrics["segformer3d_dice"] - metrics["segresnet_dice"]
    else:
        # Baseline: SegFormer3D does not exist. No two-arm delta is used for
        # the target (per avoid:); report a fixed, clearly-failing gap
        # instead of crashing, so the guardrail/target are still comparable.
        metrics["dice_gap"] = -1.0

except Exception as exc:  # never crash: degrade metrics instead
    sys.stderr.write(f"[eval_segformer3d_brats_parity] degraded run: {exc!r}\n")

print(json.dumps(metrics))

Task01_BrainTumour.tar: 0.00B [00:00, ?B/s]

Task01_BrainTumour.tar:   0%|          | 8.00k/7.09G [00:00<38:59:04, 54.2kB/s]

Task01_BrainTumour.tar:   0%|          | 7.99M/7.09G [00:00<03:22, 37.4MB/s]   

Task01_BrainTumour.tar:   0%|          | 14.2M/7.09G [00:00<03:05, 40.9MB/s]

Task01_BrainTumour.tar:   0%|          | 16.3M/7.09G [00:00<03:37, 34.9MB/s]

...

2026-09-14 22:55:03,438 - INFO - Verified 'Task01_BrainTumour.tar', md5: 240a19d752f0d9e9101544901065d872.


2026-09-14 22:55:03,446 - INFO - Downloaded: /workspace/target_repo/monai_data/Task01_BrainTumour.tar


2026-09-14 22:55:03,447 - INFO - Writing into directory: /workspace/target_repo/monai_data.


{"dice_gap": -1.0, "existing_nets_import_success_rate": 1.0, "segformer3d_dice": 0.0, "segresnet_dice": 0.0, "segformer3d_params": 0, "segresnet_params": 4702227, "segformer3d_train_time_s": 0.0, "segresnet_train_time_s": 0.0, "segformer3d_max_mem_mb": 0.0, "segresnet_max_mem_mb": 0.0}


/workspace/target_repo/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)
[eval_segformer3d_brats_parity] degraded run: RuntimeError('applying transform <monai.transforms.io.dictionary.LoadImaged object at 0x7fe7b79b9bb0>')


## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
loop: {max_iterations: 8, fix_code: true}
benchmarks:
  - name: segformer3d-brats-architecture-parity
    suite: "eval/eval_segformer3d_brats_parity.py"
    scorer: dice_gap
    baseline: dev
    metrics:
      - name: dice_gap
        role: target
        direction: max
        threshold: -0.05
      - name: existing_nets_import_success_rate
        role: guardrail
        direction: max
        threshold: 1.0
      - name: segformer3d_dice
        role: cost
        direction: max
        threshold: 0.0
      - name: segresnet_dice
        role: cost
        direction: max
        threshold: 0.0
      - name: segformer3d_params
        role: cost
        direction: min
        threshold: 10000000
      - name: segresnet_params
        role: cost
        direction: min
        threshold: 10000000
      - name: segformer3d_train_time_s
        role: cost
        direction: min
        threshold: 1800
      - name: segresnet_train_time_s
        role: cost
        direction: min
        threshold: 1800
      - name: segformer3d_max_mem_mb
        role: cost
        direction: min
        threshold: 40000
      - name: segresnet_max_mem_mb
        role: cost
        direction: min
        threshold: 40000
    policy: {guardrail_veto: true}
held_constant:
  - "same fixed 8/4 subset of the real Task01_BrainTumour training list: deterministically selected with one fixed seed for both models, identical inputs for both arms"
  - "same crop geometry: (64, 64, 64) voxels, same seed, same crop/resize pipeline applied to the real Task01_BrainTumour volumes for both models"
  - "same optimizer, learning rate and loss: Adam(lr=1e-4) with DiceLoss(sigmoid=True, squared_pred=True), identical for both models"
  - "same fixed step budget: 200 from-scratch training iterations for both models, no pretrained weights for either"
  - "same torch/numpy/monai seed across both models via monai.utils.set_determinism, single seed only, no multi-seed averaging"
  - "Task01_BrainTumour is fetched exactly once via monai.apps.DecathlonDataset(root_dir=<working dir>, task='Task01_BrainTumour', download=True) and cached in the working directory; repeat runs reuse the cache rather than re-downloading"
avoid:
  - "this is a fixed-budget, from-scratch comparison on a fixed real 8/4 Task01_BrainTumour subset -- a parity signal, not a reproduction of either paper's full training protocol or fully converged accuracy on all of BraTS"
  - "the multi-gigabyte Task01_BrainTumour download is now exercised per user_guidance; it is fetched once into the working directory and cached, so repeat invocations of this script must detect the existing cache and skip re-downloading to stay practical"
  - "no two-arm delta is used for the target: the dev baseline cannot import SegFormer3D at all, so both models are trained and scored from the feature arm and the gap is read against a fixed bound"
  - "train time, parameter counts and peak device memory are reported only as cost references, never as a pass/fail gate on model quality"
  - "no invented paper Dice numbers: only Dice computed by this script's own from-scratch training run on the real cached subset is reported"
  - "single seed only, as instructed: no seed-to-seed variance estimate exists for dice_gap, so the fixed -0.05 tolerance band remains a fixed bound rather than a noise-derived one"
compute:
  tier: gpu
  timeout_s: 5400
provenance:
  dice_gap: "user_guidance (fixed-budget from-scratch Dice comparison on the real Task01_BrainTumour 8/4 subset, no two-arm delta)"
  existing_nets_import_success_rate: "inferred -- regression guardrail for the monai/networks/nets/__init__.py edit this PR makes, since that file is shared by every existing net"
  segformer3d_dice: "user_guidance (mean validation Dice per model)"
  segresnet_dice: "user_guidance (mean validation Dice per model)"
  segformer3d_params: "user_guidance (params reported as a cost reference only)"
  segresnet_params: "user_guidance (params reported as a cost reference only)"
  segformer3d_train_time_s: "user_guidance (train time reported as a cost reference only; median-per-iteration extrapolation after a warm-up, per gpu-tier measurement practice)"
  segresnet_train_time_s: "user_guidance (train time reported as a cost reference only; median-per-iteration extrapolation after a warm-up, per gpu-tier measurement practice)"
  segformer3d_max_mem_mb: "inferred -- peak device memory via torch.cuda.max_memory_allocated, required for gpu-tier wall-clock/device measurements"
  segresnet_max_mem_mb: "inferred -- peak device memory via torch.cuda.max_memory_allocated, required for gpu-tier wall-clock/device measurements"
  held_constant: "user_guidance (same fixed 8/4 subset and seed, downloaded once via monai.apps.DecathlonDataset) refined by protocol_doc:monai/networks/nets/segformer3d.py for the channel/crop geometry"
  compute: "inferred -- timeout_s raised from 3600 to 5400 to cover the one-time Task01_BrainTumour download alongside the unchanged 200-iteration x2-model training budget"
  suite: "synthesized (R1 maturity repo: tests + CI only, no BraTS benchmark harness exists to run (a) against); loads the real cached Task01_BrainTumour subset via monai.apps.DecathlonDataset per user_guidance instead of synthetic volumes"
```